# U2T02: SimCSE Sentence Embedding Model
## Unsupervised SimCSE Pipeline, Ablation Study & Hugging Face Publishing

**Team Lead / Author:** Russel Ku (Russ)  
**Role:** Unsupervised SimCSE Lead, Unsupervised Ablation & Hugging Face Hub Publication  
**Hardware Target:** NVIDIA GeForce RTX 4070 Laptop GPU (CUDA-accelerated)  
**Base Model:** `bert-base-uncased` (110M parameters)  
**Dataset:** SNLI 100k Subset (`data/snli_train_100k.jsonl`)

---

### Overview
This notebook demonstrates the complete, end-to-end workflow for **Unsupervised SimCSE** (Gao et al., EMNLP 2021):
1. **Environment & CUDA Initialization:** Ensuring PyTorch is properly utilizing the RTX 4070 GPU.
2. **Data Pipeline:** Loading and tokenizing ~165k unique sentences from SNLI 100k with specialized collators.
3. **Architecture & Contrastive Objective:** BERT backbone with CLS pooling, MLP projection head, and InfoNCE loss using standard dropout noise as data augmentation.
4. **Model Training:** Training with AdamW, linear warmup schedule, and intermediate STS-B Dev evaluation.
5. **Unsupervised Ablation Study:** Empirically validating the necessity of independent dropout masks vs. identical masks (`same_dropout_mask=True`).
6. **STS-Benchmark Evaluation:** Computing Spearman rank correlation, Alignment ($\alpha=2$), and Uniformity ($t=2$).
7. **Hugging Face Hub Export & Verification:** Converting model to SentenceTransformers format and verifying post-upload performance.

## 1. Setup, Reproducibility & CUDA Hardware Check

In [1]:
import os
import sys
import time
import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

# Ensure project root is in sys.path
if os.path.exists("src"):
    project_root = os.path.abspath(os.getcwd())
else:
    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data import (
    build_unsupervised_dataset,
    UnsupervisedSimCSEDataset,
    UnsupDataCollator,
    load_stsb_dataset,
)
from src.models import SimCSEModel
from src.metrics import evaluate_sts_benchmark, compute_alignment, compute_uniformity
from src.train import set_seed, evaluate_on_stsb

# Reproducibility
set_seed(42)

# CUDA Device Check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[✓] Using compute device: {device}")
if torch.cuda.is_available():
    print(f"[✓] GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"[✓] VRAM Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    print(f"[✓] PyTorch CUDA Version: {torch.version.cuda}")

[✓] Using compute device: cuda
[✓] GPU Model: NVIDIA GeForce RTX 4070 Laptop GPU
[✓] VRAM Allocated: 0.00 MB
[✓] PyTorch CUDA Version: 12.4


## 2. Load & Prepare SNLI 100k Unsupervised Dataset
In Unsupervised SimCSE, each unique premise and hypothesis from SNLI is treated as an individual training instance. The positive view is generated on-the-fly by feeding the exact same tokenized input through the encoder twice with different dropout masks.

In [2]:
data_path = os.path.join(project_root, "data", "snli_train_100k.jsonl")
if not os.path.exists(data_path):
    data_path = os.path.join(project_root, "snli_train_100k.jsonl")

unique_sentences = build_unsupervised_dataset(data_path)
print(f"[✓] Extracted {len(unique_sentences):,} unique sentences from SNLI 100k.")
print("Sample sentences:")
for s in unique_sentences[:5]:
    print(f"  - {s}")

# Initialize Tokenizer and Collator
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
unsup_dataset = UnsupervisedSimCSEDataset(unique_sentences)
unsup_collator = UnsupDataCollator(tokenizer=tokenizer, max_length=64)

batch_size = 64
train_loader = DataLoader(
    unsup_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=unsup_collator,
    drop_last=True
)
print(f"[✓] DataLoader created: {len(train_loader)} batches of size {batch_size}.")

[✓] Extracted 165,528 unique sentences from SNLI 100k.
Sample sentences:
  - "Cafe Express" sign covered in graffiti.
  - # 8 for Iowa State stiff arms a Texas AM player attempting to tackle him.
  - ' The Herald' being sold and advertised at a mini-mart.
  - 'The Herald' at a mini-mart.
  - 1 guy plays the drums and makes the drums make noises that sound like a guitar strumming
[✓] DataLoader created: 2586 batches of size 64.


## 3. SimCSE Model Architecture & InfoNCE Loss

$$\ell_i = -\log \frac{e^{\text{sim}(h_i, h_i^+) / \tau}}{\sum_{j=1}^N e^{\text{sim}(h_i, h_j^+) / \tau}}$$

- **Positive pair ($h_i, h_i^+$):** The same sentence encoded with two independent dropout masks.
- **In-batch negatives ($h_j^+$):** The representations of the remaining $N-1$ sentences in the batch.
- **MLP Projection Head:** A single linear layer with `tanh` activation used during training and discarded at test time.

In [3]:
# Load STS-B Dev & Test benchmarks for evaluation
dev_records, test_records = load_stsb_dataset()
print(f"[✓] Loaded STS-B Benchmark: {len(dev_records)} Dev pairs, {len(test_records)} Test pairs.")

# Instantiate SimCSE Model
model = SimCSEModel(
    model_name_or_path=model_name,
    pooling="cls",
    temperature=0.05,
    use_mlp=True
).to(device)

print(f"[✓] Model initialized: {model_name} with CLS pooling & tau=0.05 on {device}.")

[✓] Loaded STS-B Benchmark: 1500 Dev pairs, 1379 Test pairs.


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

C:\Users\russe\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\russe\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[✓] Model initialized: bert-base-uncased with CLS pooling & tau=0.05 on cuda.


## 4. Run Unsupervised Training (Default Model)

In [4]:
import argparse
from src.train import train

parser = argparse.ArgumentParser()
args = parser.parse_args([])
args.mode = "unsup"
args.data_path = data_path
args.model_name_or_path = "bert-base-uncased"
args.output_dir = os.path.join(project_root, "checkpoints")
args.run_name = "unsup_simcse_default"
args.pooling = "cls"
args.lr = 3e-5
args.batch_size = 64
args.epochs = 1
args.temperature = 0.05
args.dropout_rate = None
args.weight_decay = 0.0
args.warmup_ratio = 0.05
args.max_length = 64
args.seed = 42
args.eval_steps = 250
args.same_dropout_mask = False
args.no_hard_negatives = False

# Execute training run
unsup_results = train(args)

Training on device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded STS-B: 1500 Dev, 1379 Test pairs.
Loading training data from c:\Users\russe\Documents\github_repo\Sentence-embedding-model\data\snli_train_100k.jsonl for mode: unsup
Built Unsupervised dataset with 165528 unique sentences.

--- Starting Training: unsup_simcse_default ---
Total Steps: 2586 | Epochs: 1 | Batch Size: 64 | LR: 3e-05


Epoch 1/1:  10%|▉         | 249/2586 [06:40<27:32,  1.41it/s, loss=0.0007, lr=2.85e-05]  


[Step 250] Dev Spearman: 77.12 (Best: -1.00)


Epoch 1/1:  17%|█▋        | 448/2586 [12:06<57:49,  1.62s/it, loss=0.0002, lr=2.61e-05]  


KeyboardInterrupt: 

## 5. Unsupervised Ablation Study: Independent vs. Same Dropout Mask

**Research Question:** Is dropout noise truly acting as a necessary minimal data augmentation?
**Hypothesis:** Setting `same_dropout_mask = True` forces identical representations $z_1 = z_2$ for both passes. Without representation variance, contrastive learning loses its augmentation signal and performance degrades significantly.

In [ ]:
from scripts.run_ablations import run_unsupervised_ablation

print("[...] Running Unsupervised Ablation (Same Dropout Mask)...")
ablation_results = run_unsupervised_ablation(data_path=data_path)

## 6. Ablation Comparison & Metrics Summary

In [ ]:
# Load run JSONs to display comparative results
runs_dir = os.path.join(project_root, "runs")
unsup_log_path = os.path.join(runs_dir, "unsup_simcse_default.json")
ablation_log_path = os.path.join(runs_dir, "ablation_unsup_same_mask.json")

with open(unsup_log_path, "r") as f:
    unsup_meta = json.load(f)
with open(ablation_log_path, "r") as f:
    ablation_meta = json.load(f)

print("=" * 85)
print(f"{'Experiment':<35} | {'Dev Spearman':<14} | {'Test Spearman':<14} | {'Alignment':<10} | {'Uniformity'}")
print("=" * 85)
u_dev = unsup_meta.get('results', {}).get('dev', {}).get('spearman', 0.0)
u_test = unsup_meta.get('results', {}).get('test', {}).get('spearman', 0.0)
u_align = unsup_meta.get('results', {}).get('test', {}).get('alignment', 0.0)
u_unif = unsup_meta.get('results', {}).get('test', {}).get('uniformity', 0.0)
a_dev = ablation_meta.get('results', {}).get('dev', {}).get('spearman', 0.0)
a_test = ablation_meta.get('results', {}).get('test', {}).get('spearman', 0.0)
a_align = ablation_meta.get('results', {}).get('test', {}).get('alignment', 0.0)
a_unif = ablation_meta.get('results', {}).get('test', {}).get('uniformity', 0.0)
print(f"{'Unsup Default (Indep Dropout)':<35} | {u_dev:<14.2f} | {u_test:<14.2f} | {u_align:<10.4f} | {u_unif:.4f}")
print(f"{'Unsup Ablation (Same Mask)':<35} | {a_dev:<14.2f} | {a_test:<14.2f} | {a_align:<10.4f} | {a_unif:.4f}")
delta_dev = u_dev - a_dev
delta_test = u_test - a_test
print("-" * 85)
print(f"[Δ Delta]: Independent dropout yields +{delta_dev:.2f} Dev and +{delta_test:.2f} Test Spearman points.")
print("=" * 85)


## 7. Hugging Face Hub Export & Verification Pipeline
Converts our trained PyTorch checkpoint into a standard `SentenceTransformers` model, generates a rich Model Card (`README.md`), and verifies reloading from Hugging Face Hub.

In [ ]:
from src.publish import export_to_sentence_transformers, verify_hf_model

best_ckpt = os.path.join(project_root, "checkpoints", "unsup_simcse_default", "best_checkpoint")
export_dir = os.path.join(project_root, "checkpoints", "unsup_simcse_default", "sentence_transformer_export")

# 1. Export locally to SentenceTransformers format
export_to_sentence_transformers(best_ckpt, export_dir, pooling="cls")
print(f"[✓] Exported model ready for Hugging Face Hub at: {export_dir}")

# 2. Example sentences test
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

st_model = SentenceTransformer(export_dir)
demo_sentences = [
    "A dog is running across the grass.",
    "A puppy plays on the lawn.",
    "A chef is cooking pasta in an Italian kitchen."
]
embs = st_model.encode(demo_sentences, normalize_embeddings=True)
sim_matrix = cosine_similarity(embs)
print("\nPairwise Cosine Similarities:")
print(f"- 'Dog running' vs 'Puppy plays': {sim_matrix[0, 1]:.4f} (High semantic similarity)")
print(f"- 'Dog running' vs 'Chef cooking': {sim_matrix[0, 2]:.4f} (Low semantic similarity)")